# 3. Clustering Non Supervisionato

Confrontiamo K-Means e clustering gerarchico agglomerativo con **single linkage** e **complete linkage**, i linkage trattati nel corso, su due feature set. La silhouette e' la metrica interna principale, ma viene letta insieme alle dimensioni dei cluster: un valore elevato puo' essere fuorviante quando il metodo isola pochi outlier in cluster quasi vuoti.

Tutti i valori di `k` da 2 a 8 vengono valutati. Tra le configurazioni K-Means, `k=2` e `k=4` sul Set A hanno silhouette quasi equivalenti; `k=4` viene mantenuto come partizione esplorativa orientata all'interpretazione, mentre `k=2` resta l'alternativa piu' parsimoniosa. Le dimensioni dei cluster e il flag euristico di squilibrio sono diagnostiche interpretative, non criteri matematici assoluti di esclusione.


In [1]:
from pathlib import Path
import os

# Make the notebook robust both when executed from the project root and from
# notebook_final.
if Path.cwd().name != "notebook_final" and (Path.cwd() / "notebook_final").exists():
    os.chdir(Path.cwd() / "notebook_final")

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
RAW_DATA_PATH = PROJECT_ROOT / "input" / "nasa_exoplanet_intelligence.csv"

import json
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

SEED = 42
N_INIT = 20
K_RANGE = range(2, 9)

processed_dir = Path("data/processed")
feature_dir = processed_dir / "feature_sets"
cluster_dir = processed_dir / "clustering"
cluster_dir.mkdir(parents=True, exist_ok=True)

X_A = pd.read_csv(feature_dir / "X_A.csv")
X_B = pd.read_csv(feature_dir / "X_B.csv")
with open(feature_dir / "feature_sets_metadata.json", encoding="utf-8") as f:
    fs_meta = json.load(f)

feature_sets = {
    "A_planetary_orbital": X_A,
    "B_planetary_orbital_stellar": X_B,
}

for name, X in feature_sets.items():
    print(f"{name}: {X.shape}")


A_planetary_orbital: (6150, 6)
B_planetary_orbital_stellar: (6150, 12)


## 3.1 Valutazione K-Means, single linkage e complete linkage


In [2]:
def cluster_diagnostics(labels):
    counts = pd.Series(labels).value_counts()
    min_size = int(counts.min())
    max_fraction = float(counts.max() / counts.sum())
    n_singletons = int((counts == 1).sum())
    min_required = max(30, int(np.ceil(0.01 * counts.sum())))
    degenerate = bool(min_size < min_required or max_fraction > 0.95)
    return {
        "min_cluster_size": min_size,
        "max_cluster_fraction": max_fraction,
        "n_singletons": n_singletons,
        "degenerate": degenerate,
        "cluster_sizes": "|".join(map(str, counts.sort_values(ascending=False).tolist())),
    }


def evaluate_kmeans(X, feature_set_name):
    rows = []
    for k in K_RANGE:
        model = KMeans(n_clusters=k, random_state=SEED, n_init=N_INIT)
        labels = model.fit_predict(X)
        rows.append({
            "feature_set": feature_set_name,
            "method": "kmeans",
            "k": k,
            "silhouette": silhouette_score(X, labels),
            "inertia": model.inertia_,
            **cluster_diagnostics(labels),
        })
    return rows


def evaluate_agglomerative(X, feature_set_name, linkage_name):
    rows = []
    for k in K_RANGE:
        model = AgglomerativeClustering(n_clusters=k, linkage=linkage_name)
        labels = model.fit_predict(X)
        rows.append({
            "feature_set": feature_set_name,
            "method": f"agglomerative_{linkage_name}",
            "k": k,
            "silhouette": silhouette_score(X, labels),
            "inertia": np.nan,
            **cluster_diagnostics(labels),
        })
    return rows


rows = []
for name, X in feature_sets.items():
    rows.extend(evaluate_kmeans(X, name))
    for linkage_name in ["single", "complete"]:
        rows.extend(evaluate_agglomerative(X, name, linkage_name))

results = pd.DataFrame(rows).sort_values("silhouette", ascending=False).reset_index(drop=True)
display(results.head(15).round(4))

print("Configurazioni con silhouette piu alta: controllo di degenerazione")
display(results.head(10)[[
    "feature_set", "method", "k", "silhouette", "cluster_sizes",
    "min_cluster_size", "max_cluster_fraction", "n_singletons", "degenerate",
]].round(4))


,feature_set,method,k,silhouette,inertia,min_cluster_size,max_cluster_fraction,n_singletons,degenerate,cluster_sizes
0,B_planetary_orbital_stellar,agglomerative_single,4,0.8228,NaN,1,0.9992,1,True,6145|2|2|1
1,B_planetary_orbital_stellar,agglomerative_single,2,0.8228,NaN,1,0.9998,1,True,6149|1
2,B_planetary_orbital_stellar,agglomerative_single,3,0.8227,NaN,1,0.9995,1,True,6147|2|1
3,A_planetary_orbital,agglomerative_single,2,0.7869,NaN,1,0.9998,1,True,6149|1
4,B_planetary_orbital_stellar,agglomerative_complete,2,0.7667,NaN,14,0.9977,0,True,6136|14
5,B_planetary_orbital_stellar,agglomerative_complete,3,0.7666,NaN,5,0.9969,0,True,6131|14|5
6,B_planetary_orbital_stellar,agglomerative_complete,4,0.7480,NaN,5,0.9932,0,True,6108|23|14|5
7,B_planetary_orbital_stellar,agglomerative_complete,5,0.7478,NaN,2,0.9932,0,True,6108|23|14|3|2
8,B_planetary_orbital_stellar,agglomerative_single,5,0.7347,NaN,1,0.9972,1,True,6133|12|2|2|1
9,B_planetary_orbital_stellar,agglomerative_single,6,0.7236,NaN,1,0.9971,2,True,6132|12|2|2|1|1


Configurazioni con silhouette piu alta: controllo di degenerazione


,feature_set,method,k,silhouette,cluster_sizes,min_cluster_size,max_cluster_fraction,n_singletons,degenerate
0,B_planetary_orbital_stellar,agglomerative_single,4,0.8228,6145|2|2|1,1,0.9992,1,True
1,B_planetary_orbital_stellar,agglomerative_single,2,0.8228,6149|1,1,0.9998,1,True
2,B_planetary_orbital_stellar,agglomerative_single,3,0.8227,6147|2|1,1,0.9995,1,True
3,A_planetary_orbital,agglomerative_single,2,0.7869,6149|1,1,0.9998,1,True
4,B_planetary_orbital_stellar,agglomerative_complete,2,0.7667,6136|14,14,0.9977,0,True
5,B_planetary_orbital_stellar,agglomerative_complete,3,0.7666,6131|14|5,5,0.9969,0,True
6,B_planetary_orbital_stellar,agglomerative_complete,4,0.7480,6108|23|14|5,5,0.9932,0,True
7,B_planetary_orbital_stellar,agglomerative_complete,5,0.7478,6108|23|14|3|2,2,0.9932,0,True
8,B_planetary_orbital_stellar,agglomerative_single,5,0.7347,6133|12|2|2|1,1,0.9972,1,True
9,B_planetary_orbital_stellar,agglomerative_single,6,0.7236,6132|12|2|2|1|1,1,0.9971,2,True


## 3.2 Grafici di selezione


In [3]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for name in feature_sets:
    sub = results[(results["feature_set"] == name) & (results["method"] == "kmeans")].sort_values("k")
    axes[0].plot(sub["k"], sub["inertia"], marker="o", label=name)

    for method, marker, linestyle in [
        ("kmeans", "o", "-"),
        ("agglomerative_single", "s", "--"),
        ("agglomerative_complete", "^", ":"),
    ]:
        sub_method = results[(results["feature_set"] == name) & (results["method"] == method)].sort_values("k")
        axes[1].plot(
            sub_method["k"],
            sub_method["silhouette"],
            marker=marker,
            linestyle=linestyle,
            label=f"{name} - {method}",
        )

axes[0].set_title("Elbow Method - KMeans inertia")
axes[0].set_xlabel("k")
axes[0].set_ylabel("Inertia")
axes[0].legend(fontsize=8)

axes[1].set_title("Silhouette: score elevati possono essere degeneri")
axes[1].set_xlabel("k")
axes[1].set_ylabel("Silhouette")
axes[1].legend(fontsize=7, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


## 3.3 Scelta della configurazione finale


In [4]:
highest_raw = results.iloc[0].to_dict()
heuristically_balanced = results[~results["degenerate"]].sort_values("silhouette", ascending=False)
best_by_balance_heuristic = heuristically_balanced.iloc[0].to_dict()

# Tutte le configurazioni KMeans con k da 2 a 8 partecipano al confronto.
# Il flag di squilibrio resta una diagnostica euristica e non filtra il pool.
selection_pool = results[results["method"] == "kmeans"].copy()
selected = selection_pool.sort_values("silhouette", ascending=False).iloc[0].to_dict()
parsimonious_k2 = selection_pool[
    (selection_pool["feature_set"] == selected["feature_set"])
    & (selection_pool["k"] == 2)
].iloc[0].to_dict()
silhouette_delta_k4_k2 = float(selected["silhouette"] - parsimonious_k2["silhouette"])

FINAL_FEATURE_SET = selected["feature_set"]
FINAL_METHOD = selected["method"]
FINAL_K = int(selected["k"])
X_final = feature_sets[FINAL_FEATURE_SET]

print("Configurazione con silhouette grezza piu alta:")
print(highest_raw)
print("\nMiglior configurazione secondo la diagnostica euristica di bilanciamento:")
print(best_by_balance_heuristic)
print("\nAlternativa KMeans parsimoniosa (k=2 sul feature set selezionato):")
print(parsimonious_k2)
print(f"Differenza silhouette k=4 meno k=2: {silhouette_delta_k4_k2:.6f}")
print("\nConfigurazione finale orientata alla interpretazione:")
print(selected)
print("\nMotivazione: k=4 ha la silhouette KMeans piu alta, ma risulta quasi equivalente a k=2; viene mantenuto per descrivere una struttura piu ricca, non per superiorita statistica.")


Configurazione con silhouette grezza piu alta:
{'feature_set': 'B_planetary_orbital_stellar', 'method': 'agglomerative_single', 'k': 4, 'silhouette': 0.8228098598643738, 'inertia': nan, 'min_cluster_size': 1, 'max_cluster_fraction': 0.9991869918699187, 'n_singletons': 1, 'degenerate': True, 'cluster_sizes': '6145|2|2|1'}

Miglior configurazione secondo la diagnostica euristica di bilanciamento:
{'feature_set': 'A_planetary_orbital', 'method': 'kmeans', 'k': 4, 'silhouette': 0.4470858036933995, 'inertia': 15253.796597646911, 'min_cluster_size': 72, 'max_cluster_fraction': 0.6591869918699187, 'n_singletons': 0, 'degenerate': False, 'cluster_sizes': '4054|1174|850|72'}

Alternativa KMeans parsimoniosa (k=2 sul feature set selezionato):
{'feature_set': 'A_planetary_orbital', 'method': 'kmeans', 'k': 2, 'silhouette': 0.44599009086717084, 'inertia': 23757.122545738493, 'min_cluster_size': 1954, 'max_cluster_fraction': 0.6822764227642276, 'n_singletons': 0, 'degenerate': False, 'cluster_sizes

## 3.4 Fit finale e PCA dei cluster


In [5]:
if FINAL_METHOD == "kmeans":
    final_model = KMeans(n_clusters=FINAL_K, random_state=SEED, n_init=N_INIT)
else:
    final_linkage = FINAL_METHOD.replace("agglomerative_", "")
    final_model = AgglomerativeClustering(n_clusters=FINAL_K, linkage=final_linkage)

FINAL_LABELS = final_model.fit_predict(X_final)
FINAL_SILHOUETTE = silhouette_score(X_final, FINAL_LABELS)

cluster_counts = pd.Series(FINAL_LABELS, name="cluster").value_counts().sort_index()
display(cluster_counts.to_frame("n_pianeti"))

pca = PCA(n_components=2, random_state=SEED)
coords = pca.fit_transform(X_final)
pca_df = pd.DataFrame({"PC1": coords[:, 0], "PC2": coords[:, 1], "cluster": FINAL_LABELS})

plt.figure(figsize=(10, 7))
sns.scatterplot(data=pca_df, x="PC1", y="PC2", hue="cluster", palette="tab10", alpha=0.65, s=22)
plt.title(f"PCA 2D - {FINAL_METHOD}, k={FINAL_K}, silhouette={FINAL_SILHOUETTE:.4f}")
plt.legend(title="cluster", bbox_to_anchor=(1.03, 1), loc="upper left")
plt.tight_layout()
plt.show()


,n_pianeti
cluster,
0,4054
1,1174
2,72
3,850


## 3.5 Dendrogrammi single e complete linkage


In [6]:
sample_idx = np.random.RandomState(SEED).choice(len(X_final), size=min(500, len(X_final)), replace=False)
sample = X_final.iloc[sample_idx]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, linkage_name in zip(axes, ["single", "complete"]):
    Z = linkage(sample, method=linkage_name)
    dendrogram(Z, truncate_mode="level", p=5, no_labels=True, ax=ax)
    ax.set_title(f"Dendrogramma {linkage_name} - {FINAL_FEATURE_SET}")
    ax.set_xlabel("Campioni / cluster")
    ax.set_ylabel("Distanza")
plt.tight_layout()
plt.show()


## 3.6 Stabilita' bootstrap


In [7]:
N_BOOT = 20
boot_silhouettes = []
for i in range(N_BOOT):
    sample = X_final.sample(frac=0.8, replace=True, random_state=i)
    model = KMeans(n_clusters=FINAL_K, random_state=i, n_init=10)
    labels = model.fit_predict(sample)
    boot_silhouettes.append(silhouette_score(sample, labels))

boot_mean = float(np.mean(boot_silhouettes))
boot_std = float(np.std(boot_silhouettes))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.boxplot(y=boot_silhouettes, color="lightblue", ax=axes[0])
axes[0].axhline(FINAL_SILHOUETTE, color="red", linestyle="--", label="full dataset")
axes[0].set_title("Bootstrap silhouette")
axes[0].legend()

axes[1].plot(range(1, N_BOOT + 1), boot_silhouettes, marker="o")
axes[1].axhline(boot_mean, color="red", linestyle="--", label=f"mean={boot_mean:.4f}")
axes[1].fill_between(
    range(1, N_BOOT + 1),
    boot_mean - boot_std,
    boot_mean + boot_std,
    color="steelblue",
    alpha=0.2,
    label=f"+/- std={boot_std:.4f}",
)
axes[1].set_title("Silhouette per bootstrap")
axes[1].set_xlabel("iterazione")
axes[1].set_ylabel("silhouette")
axes[1].legend()
plt.tight_layout()
plt.show()

print(f"Bootstrap silhouette: mean={boot_mean:.4f}, std={boot_std:.4f}")


Bootstrap silhouette: mean=0.4116, std=0.0648


## 3.7 Salvataggio clustering


In [8]:
X_clustered = X_final.copy()
X_clustered["cluster"] = FINAL_LABELS
X_clustered.to_csv(cluster_dir / "X_clustered.csv", index=False)
results.to_csv(cluster_dir / "clustering_comparison.csv", index=False)

selected_features = (
    fs_meta["feature_set_A"]
    if FINAL_FEATURE_SET == "A_planetary_orbital"
    else fs_meta["feature_set_B"]
)

def compact_result(row):
    return {
        "feature_set": row["feature_set"],
        "method": row["method"],
        "k": int(row["k"]),
        "silhouette": float(row["silhouette"]),
        "cluster_sizes": row["cluster_sizes"],
        "min_cluster_size": int(row["min_cluster_size"]),
        "max_cluster_fraction": float(row["max_cluster_fraction"]),
        "degenerate_heuristic": bool(row["degenerate"]),
    }

metadata = {
    "highest_silhouette_raw": compact_result(highest_raw),
    "best_by_balance_heuristic": compact_result(best_by_balance_heuristic),
    "parsimonious_k2_alternative": compact_result(parsimonious_k2),
    "silhouette_delta_k4_minus_k2": silhouette_delta_k4_k2,
    "final_feature_set": FINAL_FEATURE_SET,
    "final_features": selected_features,
    "final_method": FINAL_METHOD,
    "final_k": FINAL_K,
    "final_silhouette": float(FINAL_SILHOUETTE),
    "selection_rule": "compare all KMeans configurations with k=2..8; retain k=4 as an interpretation-oriented solution while reporting k=2 as the parsimonious alternative",
    "selection_reason": "k=4 has the highest KMeans silhouette, but differs from k=2 by only about 0.0011; it is retained for richer interpretation, not claimed as statistically superior",
    "balance_diagnostic_heuristic": "flag if min cluster size < max(30, 1% of n) or largest cluster fraction > 0.95; diagnostic only, not a selection filter",
    "bootstrap_mean": boot_mean,
    "bootstrap_std": boot_std,
}
with open(cluster_dir / "clustering_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4, ensure_ascii=False)

print("Artefatti clustering salvati:")
for p in sorted(cluster_dir.glob("*")):
    print("-", p)


Artefatti clustering salvati:
- data\processed\clustering\clustering_comparison.csv
- data\processed\clustering\clustering_metadata.json
- data\processed\clustering\X_clustered.csv
